# Aggregation Strategies: Choosing How Judges Reach Consensus

**Goal:** Understand the different ways to combine judge scores and when to use each strategy.

In this tutorial, you'll learn:
1. The four built-in aggregation strategies
2. How each strategy calculates final scores and confidence
3. When to use each strategy in practice
4. How to switch strategies dynamically
5. Trade-offs between different approaches

---

## The Challenge

When multiple judges evaluate the same content, they often disagree. How do we combine their opinions into a single, reliable verdict?

Different situations call for different aggregation approaches:
- **Democratic voting** when all judges are equally trusted
- **Weighted voting** when some judges are more reliable
- **Consensus requirements** when you need strong agreement
- **Statistical averaging** for numerical precision

---

## Setup

In [ ]:
# API Configuration
import os

GROQ_API_KEY = "gsk_YOUR_API_KEY_HERE"

# Or from environment
# GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [2]:
# Imports
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

from llm_jury.core.evaluator import JuryEvaluator
from llm_jury.judges.llm_judge import LLMJudge
from llm_jury.metrics.predefined import GroundednessMetric

# Import all aggregation strategies
from llm_jury.strategies.consensus import MajorityVoting, ConsensusStrategy
from llm_jury.strategies.weighted import WeightedSum, WeightedAverage

import pandas as pd
from IPython.display import display, HTML

print("All imports successful!")

All imports successful!


## Initialize Judge Panel

We'll create 5 judges with different characteristics to demonstrate how aggregation strategies handle diverse opinions.

In [3]:
print("Initializing 5-judge panel...\n")

# Judge 1: Large, capable model
judge_1 = LLMJudge(
    model=ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.0, api_key=GROQ_API_KEY),
    name="Llama-3.3-70B"
)

# Judge 2: Fast, smaller model
judge_2 = LLMJudge(
    model=ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.0, api_key=GROQ_API_KEY),
    name="Llama-3.1-8B"
)

# Judge 3: Openai model
judge_3 = LLMJudge(
    model=ChatGroq(model_name="openai/gpt-oss-120b", temperature=0.0, api_key=GROQ_API_KEY),
    name="Openai-GPT-120B"
)

# Judge 4: Another Openai variant
judge_4 = LLMJudge(
    model=ChatGroq(model_name="openai/gpt-oss-20b", temperature=0.0, api_key=GROQ_API_KEY),
    name="Openai-GPT-20B"
)

# Judge 5: Qwen model
judge_5 = LLMJudge(
    model=ChatGroq(model_name="qwen/qwen3-32b", temperature=0.0, api_key=GROQ_API_KEY),
    name="Qwen3-32B"
)

all_judges = [judge_1, judge_2, judge_3, judge_4, judge_5]

print(f"Panel initialized with {len(all_judges)} judges:")
for i, judge in enumerate(all_judges, 1):
    print(f"   {i}. {judge.name}")

Initializing 5-judge panel...

Panel initialized with 5 judges:
   1. Llama-3.3-70B
   2. Llama-3.1-8B
   3. Openai-GPT-120B
   4. Openai-GPT-20B
   5. Qwen3-32B


## Prepare Test Case

We'll use a moderately ambiguous test case that will produce some disagreement among judges.

In [4]:
# Test case: Technical documentation with some inferences
source_text = """
The /api/v2/upload endpoint accepts multipart/form-data requests with a maximum file size of 50MB.
Supported file types are: PDF, DOCX, TXT, and CSV. The endpoint returns a JSON response containing
the file ID and processing status. Authentication is required via API key in the X-API-Key header.
"""

output_text = """
You can upload files up to 50MB using the /api/v2/upload endpoint with multipart form data.
We support document formats like PDF, Word, text files, and spreadsheets. After uploading,
you'll receive a JSON response with the file ID and its processing status. Make sure to
include your API key in the request headers for authentication.
"""

context = {
    "source_text": source_text,
    "output_text": output_text
}

metric = GroundednessMetric()

print("Test case prepared.")
print("\nNote: The output contains reasonable inferences:")
print("- 'Word' instead of 'DOCX'")
print("- 'spreadsheets' instead of 'CSV'")
print("- Implies header name without specifying 'X-API-Key'")
print("\nJudges may disagree on how strict to be about these.")

Test case prepared.

Note: The output contains reasonable inferences:
- 'Word' instead of 'DOCX'
- 'spreadsheets' instead of 'CSV'
- Implies header name without specifying 'X-API-Key'

Judges may disagree on how strict to be about these.


## Strategy 1: Majority Voting

**How it works:**
- Each judge casts a vote (their score)
- The most frequent score wins
- Confidence = percentage of judges who voted for the winner

**Best for:**
- Democratic decision-making
- When all judges are equally trusted
- Categorical or discrete ratings (1-5 stars)

**Trade-offs:**
- Ignores the magnitude of disagreement
- Outliers don't affect the result
- Can have ties in small panels

In [5]:
print("\n" + "="*80)
print("STRATEGY 1: MAJORITY VOTING")
print("="*80)

# Create jury with MajorityVoting
jury_majority = JuryEvaluator(
    judges=all_judges,
    strategy=MajorityVoting()
)

# Evaluate
result_majority = jury_majority.evaluate(context, output_text, metric)

# Display results
print(f"\nFinal Score: {result_majority.final_score}/5")
print(f"Confidence: {result_majority.confidence:.2%}")
print(f"Valid: {result_majority.is_valid}")

# Show vote distribution
metadata = result_majority.manifest.metadata.get('aggregation_metadata', {})
vote_dist = metadata.get('vote_distribution', {})

print(f"\nVote Distribution:")
for score, count in sorted(vote_dist.items(), reverse=True):
    print(f"   Score {score}: {count} votes")

print(f"\nHow confidence was calculated:")
winner_votes = vote_dist.get(int(result_majority.final_score), 0)
total_votes = metadata.get('total_votes', 0)
print(f"   {winner_votes} judges voted for the winning score ({result_majority.final_score})")
print(f"   Confidence = {winner_votes}/{total_votes} = {result_majority.confidence:.2%}")


STRATEGY 1: MAJORITY VOTING

Final Score: 4.0/5
Confidence: 80.00%
Valid: True

Vote Distribution:
   Score 5: 1 votes
   Score 4: 4 votes

How confidence was calculated:
   4 judges voted for the winning score (4.0)
   Confidence = 4/5 = 80.00%


## Strategy 2: Consensus Strategy

**How it works:**
- Same as Majority Voting, but requires a minimum agreement threshold
- If agreement is below threshold, flags low confidence
- Useful for quality gates

**Best for:**
- High-stakes decisions requiring strong agreement
- Quality assurance pipelines
- Triggering human review on uncertain cases

**Trade-offs:**
- May reject valid results if judges disagree
- Threshold must be tuned for your use case
- Still returns a score even if threshold not met

In [6]:
print("\n" + "="*80)
print("STRATEGY 2: CONSENSUS STRATEGY (70% Agreement Required)")
print("="*80)

# Create jury with ConsensusStrategy requiring 70% agreement
jury_consensus = JuryEvaluator(
    judges=all_judges,
    strategy=ConsensusStrategy(threshold=0.7)
)

# Evaluate
result_consensus = jury_consensus.evaluate(context, output_text, metric)

# Display results
print(f"\nFinal Score: {result_consensus.final_score}/5")
print(f"Confidence: {result_consensus.confidence:.2%}")
print(f"Valid: {result_consensus.is_valid}")

# Check if consensus was reached
metadata = result_consensus.manifest.metadata.get('aggregation_metadata', {})
consensus_reached = metadata.get('consensus_reached', False)
threshold = metadata.get('threshold', 0.0)

print(f"\nConsensus Analysis:")
print(f"   Required Threshold: {threshold:.0%}")
print(f"   Actual Agreement: {result_consensus.confidence:.0%}")
print(f"   Consensus Reached: {consensus_reached}")

if not consensus_reached:
    print(f"\n   WARNING: Agreement below threshold!")
    print(f"   Recommendation: Route to human review")
else:
    print(f"\n   SUCCESS: Strong consensus achieved")
    print(f"   Recommendation: Auto-approve")


STRATEGY 2: CONSENSUS STRATEGY (70% Agreement Required)

Final Score: 1.0/5
Confidence: 100.00%
Valid: False

Consensus Analysis:
   Required Threshold: 70%
   Actual Agreement: 100%
   Consensus Reached: True

   SUCCESS: Strong consensus achieved
   Recommendation: Auto-approve


## Strategy 3: Weighted Sum

**How it works:**
- Each judge has a weight (default 1.0)
- Final score = sum(weight * score) / sum(weights)
- Confidence = percentage of judges that had explicit weights

**Best for:**
- When some judges are more reliable than others
- After training a reliability model on gold data
- Trusting larger/better models more

**Trade-offs:**
- Requires tuning weights
- Can overfit to specific judge characteristics
- Less democratic than voting

In [7]:
print("\n" + "="*80)
print("STRATEGY 3: WEIGHTED SUM")
print("="*80)

# Define weights: Trust larger models more
weights = {
    "Llama-3.3-70B": 2.0,    # Highest weight - large, capable
    "Llama-3.1-8B": 1.5,    # High weight - large model
    "Openai-GPT-120B": 1.5,     # High weight - good performance
    "Openai-GPT-20B": 1.0,     # Standard weight - smaller model
    "Qwen3-32B": 1.0         # Standard weight - smaller model
}

print("\nJudge Weights:")
for judge_name, weight in weights.items():
    print(f"   {judge_name}: {weight}")

# Create jury with WeightedSum
jury_weighted = JuryEvaluator(
    judges=all_judges,
    strategy=WeightedSum(weights=weights)
)

# Evaluate
result_weighted = jury_weighted.evaluate(context, output_text, metric)

# Display results
print(f"\nFinal Score: {result_weighted.final_score:.2f}/5")
print(f"Confidence: {result_weighted.confidence:.2%}")
print(f"Valid: {result_weighted.is_valid}")

# Show calculation details
metadata = result_weighted.manifest.metadata.get('aggregation_metadata', {})
total_weight = metadata.get('total_weight', 0)

print(f"\nCalculation Details:")
print(f"   Total Weight Used: {total_weight}")
print(f"\nHow the score was calculated:")
weighted_sum = 0
for score in result_weighted.manifest.individual_scores:
    judge_weight = weights.get(score.judge_id, 1.0)
    contribution = score.score * judge_weight
    weighted_sum += contribution
    print(f"   {score.judge_id}: {score.score} * {judge_weight} = {contribution:.2f}")

print(f"\n   Final Score = {weighted_sum:.2f} / {total_weight} = {result_weighted.final_score:.2f}")


STRATEGY 3: WEIGHTED SUM

Judge Weights:
   Llama-3.3-70B: 2.0
   Llama-3.1-8B: 1.5
   Openai-GPT-120B: 1.5
   Openai-GPT-20B: 1.0
   Qwen3-32B: 1.0

Final Score: 0.80/5
Confidence: 100.00%
Valid: False

Calculation Details:
   Total Weight Used: 7.0

How the score was calculated:
   Llama-3.1-8B: 0.75 * 1.5 = 1.12
   Openai-GPT-20B: 0.5 * 1.0 = 0.50
   Llama-3.3-70B: 0.75 * 2.0 = 1.50
   Openai-GPT-120B: 1.0 * 1.5 = 1.50
   Qwen3-32B: 1.0 * 1.0 = 1.00

   Final Score = 5.62 / 7.0 = 0.80


## Strategy 4: Weighted Average

**How it works:**
- Simple arithmetic mean of all scores
- Confidence = 1 / (1 + variance)
- Treats all judges equally

**Best for:**
- Continuous numerical scores
- When variance information is valuable
- Statistical analysis and reporting

**Trade-offs:**
- Sensitive to outliers
- Doesn't account for judge quality differences
- May produce non-integer scores

In [8]:
print("\n" + "="*80)
print("STRATEGY 4: WEIGHTED AVERAGE (Simple Mean)")
print("="*80)

# Create jury with WeightedAverage
jury_average = JuryEvaluator(
    judges=all_judges,
    strategy=WeightedAverage()
)

# Evaluate
result_average = jury_average.evaluate(context, output_text, metric)

# Display results
print(f"\nFinal Score: {result_average.final_score:.2f}/5")
print(f"Confidence: {result_average.confidence:.2%}")
print(f"Valid: {result_average.is_valid}")

# Show calculation details
scores = [s.score for s in result_average.manifest.individual_scores]
metadata = result_average.manifest.metadata.get('aggregation_metadata', {})
variance = metadata.get('variance', 0)

print(f"\nStatistical Analysis:")
print(f"   Individual Scores: {scores}")
print(f"   Mean: {result_average.final_score:.2f}")
print(f"   Variance: {variance:.4f}")
print(f"\nHow confidence was calculated:")
print(f"   Confidence = 1 / (1 + variance)")
print(f"   Confidence = 1 / (1 + {variance:.4f}) = {result_average.confidence:.2%}")
print(f"\nInterpretation:")
if variance < 0.5:
    print(f"   Low variance - judges mostly agree")
elif variance < 1.0:
    print(f"   Moderate variance - some disagreement")
else:
    print(f"   High variance - significant disagreement")


STRATEGY 4: WEIGHTED AVERAGE (Simple Mean)

Final Score: 0.80/5
Confidence: 96.62%
Valid: False

Statistical Analysis:
   Individual Scores: [0.75, 0.5, 0.75, 1.0, 1.0]
   Mean: 0.80
   Variance: 0.0350

How confidence was calculated:
   Confidence = 1 / (1 + variance)
   Confidence = 1 / (1 + 0.0350) = 96.62%

Interpretation:
   Low variance - judges mostly agree


## Side-by-Side Comparison

Let's compare all four strategies on the same input to see how they differ.

In [9]:
# Compile results from all strategies
comparison_data = [
    {
        'Strategy': 'Majority Voting',
        'Final Score': f"{result_majority.final_score}/5",
        'Confidence': f"{result_majority.confidence:.1%}",
        'Valid': result_majority.is_valid,
        'Best For': 'Democratic, equal trust'
    },
    {
        'Strategy': 'Consensus (70%)',
        'Final Score': f"{result_consensus.final_score}/5",
        'Confidence': f"{result_consensus.confidence:.1%}",
        'Valid': result_consensus.is_valid,
        'Best For': 'Quality gates, high stakes'
    },
    {
        'Strategy': 'Weighted Sum',
        'Final Score': f"{result_weighted.final_score:.2f}/5",
        'Confidence': f"{result_weighted.confidence:.1%}",
        'Valid': result_weighted.is_valid,
        'Best For': 'Trust some judges more'
    },
    {
        'Strategy': 'Weighted Average',
        'Final Score': f"{result_average.final_score:.2f}/5",
        'Confidence': f"{result_average.confidence:.1%}",
        'Valid': result_average.is_valid,
        'Best For': 'Statistical analysis'
    }
]

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("STRATEGY COMPARISON")
print("="*80 + "\n")
display(comparison_df)

print("\nKey Observations:")
print("1. Different strategies can produce different final scores")
print("2. Confidence means different things in each strategy")
print("3. Weighted strategies may give fractional scores")
print("4. Consensus strategy adds threshold checking")


STRATEGY COMPARISON



,Strategy,Final Score,Confidence,Valid,Best For
0,Majority Voting,4.0/5,80.0%,True,"Democratic, equal trust"
1,Consensus (70%),1.0/5,100.0%,False,"Quality gates, high stakes"
2,Weighted Sum,0.80/5,100.0%,False,Trust some judges more
3,Weighted Average,0.80/5,96.6%,False,Statistical analysis



Key Observations:
1. Different strategies can produce different final scores
2. Confidence means different things in each strategy
3. Weighted strategies may give fractional scores
4. Consensus strategy adds threshold checking


## Individual Judge Scores

Let's examine what each judge actually said to understand why strategies differ.

In [10]:
# Extract individual scores (same across all strategies)
judge_scores = []
for score in result_majority.manifest.individual_scores:
    judge_scores.append({
        'Judge': score.judge_id,
        'Score': score.score,
        'Weight (if used)': weights.get(score.judge_id, 1.0),
        'Key Point': score.reasoning.split('.')[0][:80] + '...'
    })

judge_df = pd.DataFrame(judge_scores)

print("\n" + "="*80)
print("INDIVIDUAL JUDGE SCORES")
print("="*80 + "\n")
display(judge_df)

# Calculate how each strategy would use these scores
scores = [s['Score'] for s in judge_scores]
print(f"\nRaw Scores: {scores}")
print(f"\nHow each strategy processes these:")
print(f"   Majority Voting: Picks most frequent = {max(set(scores), key=scores.count)}")
print(f"   Weighted Average: Calculates mean = {sum(scores)/len(scores):.2f}")
weighted_calc = sum(s['Score'] * weights.get(s['Judge'], 1.0) for s in judge_scores) / sum(weights.values())
print(f"   Weighted Sum: Uses weights = {weighted_calc:.2f}")


INDIVIDUAL JUDGE SCORES



,Judge,Score,Weight (if used),Key Point
0,Llama-3.1-8B,4.0,1.5,The model output is mostly supported by the so...
1,Llama-3.3-70B,4.0,2.0,The model output is mostly supported by the so...
2,Openai-GPT-20B,4.0,1.0,The model output accurately reflects the key f...
3,Openai-GPT-120B,5.0,1.5,Every statement in the model output is directl...
4,Qwen3-32B,4.0,1.0,The Model Output is mostly supported by the So...



Raw Scores: [4.0, 4.0, 4.0, 5.0, 4.0]

How each strategy processes these:
   Majority Voting: Picks most frequent = 4.0
   Weighted Average: Calculates mean = 4.20
   Weighted Sum: Uses weights = 4.21


## Dynamic Strategy Switching

You can change strategies on the fly without recreating the jury.

In [11]:
print("\n" + "="*80)
print("DYNAMIC STRATEGY SWITCHING")
print("="*80)

# Create a jury with one strategy
dynamic_jury = JuryEvaluator(
    judges=all_judges,
    strategy=MajorityVoting()
)

print(f"\nInitial strategy: {dynamic_jury.strategy.__class__.__name__}")

# Evaluate with initial strategy
result1 = dynamic_jury.evaluate(context, output_text, metric)
print(f"Result: {result1.final_score}/5 (confidence: {result1.confidence:.1%})")

# Switch to a different strategy
print("\nSwitching to Weighted Average...")
dynamic_jury.set_strategy(WeightedAverage())
result2 = dynamic_jury.evaluate(context, output_text, metric)
print(f"Result: {result2.final_score:.2f}/5 (confidence: {result2.confidence:.1%})")

# Switch again
print("\nSwitching to Consensus Strategy (80% threshold)...")
dynamic_jury.set_strategy(ConsensusStrategy(threshold=0.8))
result3 = dynamic_jury.evaluate(context, output_text, metric)
print(f"Result: {result3.final_score}/5 (confidence: {result3.confidence:.1%})")
metadata = result3.manifest.metadata.get('aggregation_metadata', {})
print(f"Consensus reached: {metadata.get('consensus_reached', False)}")

print("\nUse case: Start with Majority Voting, switch to Consensus for critical decisions")


DYNAMIC STRATEGY SWITCHING

Initial strategy: MajorityVoting
Result: 4.0/5 (confidence: 40.0%)

Switching to Weighted Average...
Result: 0.85/5 (confidence: 98.5%)

Switching to Consensus Strategy (80% threshold)...
Result: 1.0/5 (confidence: 80.0%)
Consensus reached: True

Use case: Start with Majority Voting, switch to Consensus for critical decisions


## Production Recommendations

Best practices for using aggregation strategies in production systems.

In [12]:
recommendations = """
PRODUCTION BEST PRACTICES
================================================================================

1. START SIMPLE, ITERATE
   - Begin with Majority Voting
   - Collect data on judge performance
   - Upgrade to weighted strategies once you have evidence

2. USE CONSENSUS FOR QUALITY GATES
   - Set thresholds based on your risk tolerance
   - Route low-confidence cases to human review
   - Track consensus rates over time

3. TUNE WEIGHTS WITH DATA
   - Don't guess at weights
   - Use gold-labeled test sets
   - Measure correlation with human judgment
   - Re-tune periodically as models improve

4. MONITOR VARIANCE
   - High variance = judges disagree (ambiguous case)
   - Low variance = judges agree (clear case)
   - Use variance to trigger different workflows

5. COMBINE STRATEGIES
   - Example: Use Weighted Sum for score, Consensus for gating
   - Example: Majority Voting for speed, Weighted for accuracy
   - Switch strategies based on content type

6. HANDLE TIES GRACEFULLY
   - Small panels can tie frequently
   - Use odd number of judges (3, 5, 7)
   - Or have a tie-breaking judge

7. COST-QUALITY TRADE-OFFS
   - Weighted Sum: Use fewer expensive judges with higher weights
   - Majority Voting: Balance cheap and expensive judges
   - Consensus: May need more judges to reach threshold

8. A/B TEST STRATEGIES
   - Run parallel evaluations with different strategies
   - Compare to human gold labels
   - Measure latency, cost, and accuracy

================================================================================
"""

print(recommendations)


PRODUCTION BEST PRACTICES

1. START SIMPLE, ITERATE
   - Begin with Majority Voting
   - Collect data on judge performance
   - Upgrade to weighted strategies once you have evidence

2. USE CONSENSUS FOR QUALITY GATES
   - Set thresholds based on your risk tolerance
   - Route low-confidence cases to human review
   - Track consensus rates over time

3. TUNE WEIGHTS WITH DATA
   - Don't guess at weights
   - Use gold-labeled test sets
   - Measure correlation with human judgment
   - Re-tune periodically as models improve

4. MONITOR VARIANCE
   - High variance = judges disagree (ambiguous case)
   - Low variance = judges agree (clear case)
   - Use variance to trigger different workflows

5. COMBINE STRATEGIES
   - Example: Use Weighted Sum for score, Consensus for gating
   - Example: Majority Voting for speed, Weighted for accuracy
   - Switch strategies based on content type

6. HANDLE TIES GRACEFULLY
   - Small panels can tie frequently
   - Use odd number of judges (3, 5, 7)
   

## Key Takeaways

### Summary of Aggregation Strategies

**Majority Voting:**
- Democratic, treats all judges equally
- Best for discrete ratings
- Confidence = vote percentage
- Simple and intuitive

**Consensus Strategy:**
- Adds threshold requirement to Majority Voting
- Best for quality gates
- Flags low-confidence cases
- Enables human-in-the-loop workflows

**Weighted Sum:**
- Trusts some judges more than others
- Best when you have reliability data
- Requires weight tuning
- More sophisticated but less transparent

**Weighted Average:**
- Simple arithmetic mean
- Best for statistical analysis
- Provides variance information
- Sensitive to outliers

### When to Use Each

- **Starting out:** Majority Voting
- **Production RAG:** Consensus Strategy
- **Research/benchmarking:** Weighted Average
- **After training reliability model:** Weighted Sum
- **High stakes:** Consensus Strategy with high threshold
- **Speed matters:** Majority Voting with fewer judges

---

## Next Steps

Now that you understand aggregation strategies:
- **05_Custom_Metrics.ipynb**: Create domain-specific evaluation criteria
- **06_Agentic_Shield.ipynb**: Protect AI agents from hallucinations
- Experiment with different strategies on your own data
- Measure which strategy best matches your human judgments

---

**Happy Aggregating!**